In [3]:

import pandas as pd

# Load financial transaction dataset
file_path = r"C:\Users\ritik\OneDrive\Documents\Python Automation File\archive (4)\indian_banking_transactions.csv"

df = pd.read_csv(file_path)

# Display basic dataset information
print("FINANCIAL DATASET")
print("-----------------")

print("Total Records:", len(df))
print("Total Columns:", len(df.columns))

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Records:")
print(df.head())


FINANCIAL DATASET
-----------------
Total Records: 550000
Total Columns: 20

Column Names:
['transaction_id', 'customer_id', 'transaction_date', 'transaction_time', 'account_type', 'transaction_type', 'transaction_amount', 'transaction_direction', 'account_balance', 'merchant_category', 'state', 'credit_score', 'has_loan', 'loan_type', 'emi_amount', 'transaction_status', 'channel', 'kyc_status', 'is_fraud', 'transaction_hour']

First 5 Records:
  transaction_id customer_id transaction_date transaction_time   account_type  \
0   TXN000000001  CUST015796       2019-01-01            15:28        Current   
1   TXN000000002  CUST000861       2019-01-01            03:00        Current   
2   TXN000000003  CUST076821       2019-01-01            18:03  Fixed Deposit   
3   TXN000000004  CUST054887       2019-01-01            08:03        Savings   
4   TXN000000005  CUST006266       2019-01-01            14:23  Fixed Deposit   

  transaction_type  transaction_amount transaction_direction  ac

In [4]:

# STEP 2: DATA QUALITY ASSESSMENT

print("\n========== DATA QUALITY REPORT ==========")

# 1. Check missing values
print("\n1. MISSING VALUES")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 2. Check duplicate transaction IDs
print("\n2. DUPLICATE TRANSACTION IDs")
print(df["transaction_id"].duplicated().sum())

# 3. Check duplicate complete records
print("\n3. DUPLICATE ROWS")
print(df.duplicated().sum())

# 4. Check transaction status counts
print("\n4. TRANSACTION STATUS")
print(df["transaction_status"].value_counts())

# 5. Check transaction direction
print("\n5. TRANSACTION DIRECTION")
print(df["transaction_direction"].value_counts())

# 6. Check fraud labels
print("\n6. FRAUD LABELS")
print(df["is_fraud"].value_counts())

# 7. Check transaction amount
print("\n7. TRANSACTION AMOUNT SUMMARY")
print(df["transaction_amount"].describe())

print("\n========== CHECKING COMPLETED ==========")



========== DATA QUALITY REPORT ==========

1. MISSING VALUES
loan_type    377134
dtype: int64

2. DUPLICATE TRANSACTION IDs
0

3. DUPLICATE ROWS
0

4. TRANSACTION STATUS
transaction_status
Success     506247
Failed       21905
Reversed     11019
Pending      10829
Name: count, dtype: int64

5. TRANSACTION DIRECTION
transaction_direction
Debit     370394
Credit    179606
Name: count, dtype: int64

6. FRAUD LABELS
is_fraud
0    545127
1      4873
Name: count, dtype: int64

7. TRANSACTION AMOUNT SUMMARY
count    5.500000e+05
mean     2.990709e+04
std      1.396101e+05
min      2.400000e+00
25%      7.630375e+02
50%      2.035735e+03
75%      8.557497e+03
max      1.000000e+07
Name: transaction_amount, dtype: float64

========== CHECKING COMPLETED ==========


In [5]:

# STEP 3: FINANCIAL DATA VALIDATION

print("\n========== FINANCIAL DATA VALIDATION ==========")

# 1. Check invalid transaction dates
invalid_dates = pd.to_datetime(
    df["transaction_date"], errors="coerce"
).isna().sum()

print("\nInvalid Transaction Dates:", invalid_dates)

# 2. Check zero or negative transaction amounts
invalid_amounts = (df["transaction_amount"] <= 0).sum()

print("Invalid Transaction Amounts:", invalid_amounts)

# 3. Check negative account balances
negative_balances = (df["account_balance"] < 0).sum()

print("Negative Account Balances:", negative_balances)

# 4. Check invalid credit scores
invalid_credit_scores = (
    ~df["credit_score"].between(300, 900)
).sum()

print("Invalid Credit Scores:", invalid_credit_scores)

# 5. Check transaction hours
invalid_hours = (
    ~df["transaction_hour"].between(0, 23)
).sum()

print("Invalid Transaction Hours:", invalid_hours)

# 6. Check loan information consistency
missing_loan_type = (
    (df["has_loan"] == 1) & (df["loan_type"].isna())
).sum()

print("Loan Records Missing Loan Type:", missing_loan_type)

# 7. Check binary fields
invalid_fraud_flags = (
    ~df["is_fraud"].isin([0, 1])
).sum()

print("Invalid Fraud Flags:", invalid_fraud_flags)

print("\n========== VALIDATION COMPLETED ==========")



========== FINANCIAL DATA VALIDATION ==========

Invalid Transaction Dates: 0
Invalid Transaction Amounts: 0
Negative Account Balances: 0
Invalid Credit Scores: 0
Invalid Transaction Hours: 0
Loan Records Missing Loan Type: 19422
Invalid Fraud Flags: 0

========== VALIDATION COMPLETED ==========


In [6]:

# STEP 4: DATA CLEANING & EXCEPTION HANDLING

import os

print("\n========== DATA CLEANING ==========")

# Create output folders
base_path = r"C:\Users\ritik\OneDrive\Documents\Python Automation File\archive (4)"

processed_path = os.path.join(base_path, "processed")
reports_path = os.path.join(base_path, "reports")

os.makedirs(processed_path, exist_ok=True)
os.makedirs(reports_path, exist_ok=True)

# Create a copy of original dataset
clean_df = df.copy()

# Convert transaction date into datetime format
clean_df["transaction_date"] = pd.to_datetime(
    clean_df["transaction_date"],
    errors="coerce"
)

# Identify loan records with missing loan type
exception_mask = (
    (clean_df["has_loan"] == 1) &
    (clean_df["loan_type"].isna())
)

# Create exception report
exceptions = clean_df.loc[exception_mask].copy()

exceptions["exception_reason"] = "Loan type missing despite has_loan = 1"

# Save validated dataset
clean_file = os.path.join(
    processed_path,
    "validated_transactions.csv"
)

clean_df.to_csv(clean_file, index=False)

# Save exception report
exception_file = os.path.join(
    reports_path,
    "data_quality_exceptions.csv"
)

exceptions.to_csv(exception_file, index=False)

print("Original Records:", len(df))
print("Validated Records:", len(clean_df))
print("Exception Records:", len(exceptions))

print("\nValidated Dataset Saved:", clean_file)
print("Exception Report Saved:", exception_file)

print("\n========== CLEANING COMPLETED ==========")



========== DATA CLEANING ==========
Original Records: 550000
Validated Records: 550000
Exception Records: 19422

Validated Dataset Saved: C:\Users\ritik\OneDrive\Documents\Python Automation File\archive (4)\processed\validated_transactions.csv
Exception Report Saved: C:\Users\ritik\OneDrive\Documents\Python Automation File\archive (4)\reports\data_quality_exceptions.csv

========== CLEANING COMPLETED ==========


In [7]:

# STEP 5: FINANCIAL TRANSACTION ANALYSIS

print("\n========== FINANCIAL ANALYSIS ==========")

# 1. Total transaction amount
total_amount = clean_df["transaction_amount"].sum()

# 2. Total number of transactions
total_transactions = len(clean_df)

# 3. Transaction status summary
status_summary = clean_df.groupby(
    "transaction_status"
).agg(
    transaction_count=("transaction_id", "count"),
    total_amount=("transaction_amount", "sum")
).reset_index()

# 4. Debit and Credit summary
direction_summary = clean_df.groupby(
    "transaction_direction"
).agg(
    transaction_count=("transaction_id", "count"),
    total_amount=("transaction_amount", "sum")
).reset_index()

# 5. Payment mode summary
payment_summary = clean_df.groupby(
    "payment_mode"
).agg(
    transaction_count=("transaction_id", "count"),
    total_amount=("transaction_amount", "sum")
).reset_index()

# 6. Monthly transaction summary
clean_df["month"] = clean_df["transaction_date"].dt.to_period("M").astype(str)

monthly_summary = clean_df.groupby(
    "month"
).agg(
    transaction_count=("transaction_id", "count"),
    total_amount=("transaction_amount", "sum")
).reset_index()

# Display results
print("\nTotal Transactions:", total_transactions)
print("Total Transaction Amount:", round(total_amount, 2))

print("\nTRANSACTION STATUS SUMMARY")
print(status_summary)

print("\nDEBIT VS CREDIT SUMMARY")
print(direction_summary)

print("\nPAYMENT MODE SUMMARY")
print(payment_summary)

print("\nMONTHLY TRANSACTION SUMMARY")
print(monthly_summary.head(12))

print("\n========== ANALYSIS COMPLETED ==========")



========== FINANCIAL ANALYSIS ==========


KeyError: 'payment_mode'

In [ ]:
print("\nAVAILABLE COLUMNS:")
print(clean_df.columns.tolist())


AVAILABLE COLUMNS:
['transaction_id', 'customer_id', 'transaction_date', 'transaction_time', 'account_type', 'transaction_type', 'transaction_amount', 'transaction_direction', 'account_balance', 'merchant_category', 'state', 'credit_score', 'has_loan', 'loan_type', 'emi_amount', 'transaction_status', 'channel', 'kyc_status', 'is_fraud', 'transaction_hour']


In [ ]:
# 5. Channel Summary
channel_summary = clean_df.groupby("channel").agg(
    transaction_count=("transaction_id", "count"),
    total_amount=("transaction_amount", "sum")
).reset_index()

print("\nCHANNEL SUMMARY")
print(channel_summary)


CHANNEL SUMMARY
        channel  transaction_count  total_amount
0           API              38635  1.169149e+09
1           ATM              66177  1.960455e+09
2        Branch              54957  1.606088e+09
3    Mobile_App             208368  6.244031e+09
4  POS_Terminal              60853  1.807224e+09
5           Web             121010  3.661956e+09


In [ ]:
# STEP 6: SUCCESSFUL TRANSACTION ANALYSIS

print("\n========== SUCCESSFUL TRANSACTION ANALYSIS ==========")

successful_df = clean_df[
    clean_df["transaction_status"].astype(str).str.strip().str.lower() == "success"
].copy()

print("Total Successful Transactions:", len(successful_df))

print(
    "Successful Transaction Amount:",
    round(successful_df["transaction_amount"].sum(), 2)
)

print("\nSUCCESSFUL TRANSACTIONS BY CHANNEL")

successful_channel = successful_df.groupby("channel").agg(
    transaction_count=("transaction_id", "count"),
    total_amount=("transaction_amount", "sum")
).reset_index()

print(successful_channel)

print("\n========== ANALYSIS COMPLETED ==========")


========== SUCCESSFUL TRANSACTION ANALYSIS ==========
Total Successful Transactions: 506247
Successful Transaction Amount: 15091797022.11

SUCCESSFUL TRANSACTIONS BY CHANNEL
        channel  transaction_count  total_amount
0           API              35499  1.076819e+09
1           ATM              60945  1.793281e+09
2        Branch              50560  1.475994e+09
3    Mobile_App             191817  5.719476e+09
4  POS_Terminal              55983  1.656214e+09
5           Web             111443  3.370013e+09

========== ANALYSIS COMPLETED ==========


In [ ]:
# STEP 7: FRAUD TRANSACTION ANALYSIS

print("\n========== FRAUD ANALYSIS ==========")

fraud_df = clean_df[clean_df["is_fraud"] == 1].copy()

non_fraud_df = clean_df[clean_df["is_fraud"] == 0].copy()

print("Total Fraud Transactions:", len(fraud_df))
print("Total Non-Fraud Transactions:", len(non_fraud_df))

print(
    "Fraud Transaction Amount:",
    round(fraud_df["transaction_amount"].sum(), 2)
)

print("\nFRAUD TRANSACTIONS BY CHANNEL")

fraud_channel = fraud_df.groupby("channel").agg(
    fraud_count=("transaction_id", "count"),
    fraud_amount=("transaction_amount", "sum")
).reset_index()

print(fraud_channel)

print("\nFRAUD TRANSACTIONS BY MERCHANT CATEGORY")

fraud_merchant = fraud_df.groupby("merchant_category").agg(
    fraud_count=("transaction_id", "count"),
    fraud_amount=("transaction_amount", "sum")
).reset_index()

print(fraud_merchant)

print("\n========== FRAUD ANALYSIS COMPLETED ==========")


========== FRAUD ANALYSIS ==========
Total Fraud Transactions: 4873
Total Non-Fraud Transactions: 545127
Fraud Transaction Amount: 582061390.09

FRAUD TRANSACTIONS BY CHANNEL
        channel  fraud_count  fraud_amount
0           API          346  4.971249e+07
1           ATM          561  6.061427e+07
2        Branch          520  6.101485e+07
3    Mobile_App         1809  2.090801e+08
4  POS_Terminal          557  7.155019e+07
5           Web         1080  1.300895e+08

FRAUD TRANSACTIONS BY MERCHANT CATEGORY
   merchant_category  fraud_count  fraud_amount
0         E-Commerce          502   62176206.78
1          Education          236   23310132.14
2      Entertainment          300   41348895.78
3      Food & Dining          471   62533954.71
4               Fuel          201   31093754.31
5         Government          214   24119089.34
6         Healthcare          300   28677050.94
7          Insurance          191   18081057.78
8         Investment          252   30481526.99
9 

In [ ]:
 pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# STEP 8: AUTOMATED EXCEL REPORT

print("\n========== EXCEL REPORT GENERATION ==========")

excel_file = os.path.join(
    reports_path,
    "Financial_Analysis_Report.xlsx"
)

summary_df = pd.DataFrame({
    "Metric": [
        "Total Transactions",
        "Successful Transactions",
        "Fraud Transactions",
        "Non-Fraud Transactions",
        "Total Transaction Amount",
        "Successful Transaction Amount",
        "Fraud Transaction Amount"
    ],
    "Value": [
        len(clean_df),
        len(successful_df),
        len(fraud_df),
        len(non_fraud_df),
        round(clean_df["transaction_amount"].sum(), 2),
        round(successful_df["transaction_amount"].sum(), 2),
        round(fraud_df["transaction_amount"].sum(), 2)
    ]
})

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:

    summary_df.to_excel(writer, sheet_name="Executive Summary", index=False)

    status_summary.to_excel(writer, sheet_name="Transaction Status", index=False)

    direction_summary.to_excel(writer, sheet_name="Debit Credit", index=False)

    channel_summary.to_excel(writer, sheet_name="Channel Analysis", index=False)

    monthly_summary.to_excel(writer, sheet_name="Monthly Analysis", index=False)

    successful_channel.to_excel(writer, sheet_name="Successful Analysis", index=False)

    fraud_channel.to_excel(writer, sheet_name="Fraud by Channel", index=False)

    fraud_merchant.to_excel(writer, sheet_name="Fraud by Merchant", index=False)

    exceptions.to_excel(writer, sheet_name="Data Exceptions", index=False)

print("Excel Report Generated Successfully!")
print("Report Location:", excel_file)

print("\n========== REPORT GENERATION COMPLETED ==========")


========== EXCEL REPORT GENERATION ==========
Excel Report Generated Successfully!
Report Location: C:\Users\ritik\OneDrive\Documents\Python Automation File\archive (4)\reports\Financial_Analysis_Report.xlsx

========== REPORT GENERATION COMPLETED ==========


In [ ]:
# CREATE MONTHLY SUMMARY

clean_df["month"] = clean_df["transaction_date"].dt.to_period("M").astype(str)

monthly_summary = clean_df.groupby("month").agg(
    transaction_count=("transaction_id", "count"),
    total_amount=("transaction_amount", "sum")
).reset_index()

print("Monthly Summary Created Successfully!")
print(monthly_summary.head())

Monthly Summary Created Successfully!
     month  transaction_count  total_amount
0  2019-01               9406  2.495359e+08
1  2019-02               8220  2.431736e+08
2  2019-03               9290  2.606286e+08
3  2019-04               8990  2.736464e+08
4  2019-05               9425  2.743727e+08


In [ ]:
# STEP 9: EXCEL FORMATTING

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

workbook = load_workbook(excel_file)

for sheet in workbook.worksheets:

    # Header formatting
    for cell in sheet[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill(
            start_color="1F4E78",
            end_color="1F4E78",
            fill_type="solid"
        )
        cell.alignment = Alignment(horizontal="center")

    # Adjust column widths
    for column in sheet.columns:
        max_length = 0
        column_letter = column[0].column_letter

        for cell in column:
            if cell.value is not None:
                max_length = max(max_length, len(str(cell.value)))

        sheet.column_dimensions[column_letter].width = max_length + 4

    # Freeze header
    sheet.freeze_panes = "A2"

    # Add filters
    sheet.auto_filter.ref = sheet.dimensions

workbook.save(excel_file)

print("Excel Formatting Completed Successfully!")
print("Updated Report:", excel_file)

Excel Formatting Completed Successfully!
Updated Report: C:\Users\ritik\OneDrive\Documents\Python Automation File\archive (4)\reports\Financial_Analysis_Report.xlsx


In [ ]:
# STEP 10: EXCEL CHART CREATION

from openpyxl.chart import BarChart, Reference

workbook = load_workbook(excel_file)

sheet = workbook["Channel Analysis"]

chart = BarChart()
chart.title = "Transactions by Channel"
chart.x_axis.title = "Transaction Channel"
chart.y_axis.title = "Number of Transactions"
chart.style = 10

data = Reference(sheet, min_col=2, min_row=1, max_row=sheet.max_row)
categories = Reference(sheet, min_col=1, min_row=2, max_row=sheet.max_row)

chart.add_data(data, titles_from_data=True)
chart.set_categories(categories)

sheet.add_chart(chart, "E2")

workbook.save(excel_file)

print("Channel Analysis Chart Added Successfully!")

Channel Analysis Chart Added Successfully!


In [ ]:
# CHART 2: FRAUD BY CHANNEL

workbook = load_workbook(excel_file)

sheet = workbook["Fraud by Channel"]

chart = BarChart()
chart.title = "Fraud Transactions by Channel"
chart.x_axis.title = "Channel"
chart.y_axis.title = "Fraud Count"
chart.style = 10

data = Reference(
    sheet,
    min_col=2,
    min_row=1,
    max_row=sheet.max_row
)

categories = Reference(
    sheet,
    min_col=1,
    min_row=2,
    max_row=sheet.max_row
)

chart.add_data(data, titles_from_data=True)
chart.set_categories(categories)

sheet.add_chart(chart, "E2")

workbook.save(excel_file)

print("Fraud Analysis Chart Added Successfully!")

Fraud Analysis Chart Added Successfully!


In [ ]:
from openpyxl.chart import LineChart

In [ ]:
# CHART 3: MONTHLY TRANSACTIONS

workbook = load_workbook(excel_file)

sheet = workbook["Monthly Analysis"]

chart = LineChart()
chart.title = "Monthly Transaction Trend"
chart.x_axis.title = "Month"
chart.y_axis.title = "Number of Transactions"
chart.style = 13

data = Reference(
    sheet,
    min_col=2,
    min_row=1,
    max_row=sheet.max_row
)

categories = Reference(
    sheet,
    min_col=1,
    min_row=2,
    max_row=sheet.max_row
)

chart.add_data(data, titles_from_data=True)
chart.set_categories(categories)

sheet.add_chart(chart, "E2")

workbook.save(excel_file)

print("Monthly Transaction Chart Added Successfully!")

Monthly Transaction Chart Added Successfully!


In [ ]:
# STEP 11: AUTOMATION LOGGING

import logging

log_file = os.path.join(reports_path, "automation.log")

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

logging.info("Financial data processing completed.")
logging.info("Data validation completed.")
logging.info("Fraud analysis completed.")
logging.info("Excel report generated successfully.")
logging.info("All three Excel charts generated successfully.")

print("Automation Logging Completed!")
print("Log File:", log_file)

Automation Logging Completed!
Log File: C:\Users\ritik\OneDrive\Documents\Python Automation File\archive (4)\reports\automation.log


In [ ]:
pip install pytest


   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   ---------------------------------------- 3/3 [pytest]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!python -m pytest --version

pytest 9.2.0


In [ ]:
# STEP 13: FIRST PYTEST TEST

def test_positive_transaction_amounts():
    assert (clean_df["transaction_amount"] > 0).all()

print("Test function created successfully!")

Test function created successfully!


In [13]:
%pip install ipytest

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# STEP 15: RUN FIRST PYTEST

import ipytest
ipytest.autoconfig()

ipytest.run()

.                                                                                            [100%]
1 passed in 0.02s


<ExitCode.OK: 0>

In [ ]:
# STEP 16: FRAUD FLAG TEST

def test_fraud_flags_are_valid():
    assert clean_df["is_fraud"].isin([0, 1]).all()

print("Fraud validation test created successfully!")

Fraud validation test created successfully!


In [ ]:
# STEP 17: RUN ALL PYTEST CASES

ipytest.run()

..                                                                                           [100%]
2 passed in 0.02s


<ExitCode.OK: 0>

In [ ]:
# STEP 18: TRANSACTION DATE TEST

def test_transaction_dates_are_valid():
    assert clean_df["transaction_date"].notna().all()

print("Transaction date test created successfully!")

Transaction date test created successfully!


In [ ]:
# STEP 19: RUN ALL TEST CASES

ipytest.run()

...                                                                                          [100%]
3 passed in 0.01s


<ExitCode.OK: 0>

In [ ]:
# STEP 20: ACCOUNT BALANCE TEST

def test_account_balance_is_valid():
    assert (clean_df["account_balance"] >= 0).all()

print("Account balance test created successfully!")

Account balance test created successfully!


In [ ]:
# STEP 21: RUN ALL TEST CASES

ipytest.run()

....                                                                                         [100%]
4 passed in 0.02s


<ExitCode.OK: 0>

In [ ]:
# STEP 22: CREDIT SCORE TEST

def test_credit_score_is_valid():
    assert clean_df["credit_score"].between(300, 900).all()

print("Credit score test created successfully!")

Credit score test created successfully!


In [ ]:
# STEP 23: RUN ALL TEST CASES

ipytest.run()

.....                                                                                        [100%]
5 passed in 0.03s


<ExitCode.OK: 0>

In [ ]:
# STEP 24: TRANSACTION ID TEST

def test_transaction_ids_are_not_missing():
    assert clean_df["transaction_id"].notna().all()

print("Transaction ID test created successfully!")

Transaction ID test created successfully!


In [ ]:
# STEP 25: RUN ALL TEST CASES

ipytest.run()

......                                                                                       [100%]
6 passed in 0.04s


<ExitCode.OK: 0>

In [ ]:
# STEP 26: DUPLICATE TRANSACTION ID TEST

def test_transaction_ids_are_unique():
    assert clean_df["transaction_id"].is_unique

print("Duplicate transaction ID test created successfully!")

Duplicate transaction ID test created successfully!


In [ ]:
# STEP 25: RUN ALL TEST CASES

ipytest.run()

.......                                                                                      [100%]
7 passed in 0.11s


<ExitCode.OK: 0>

In [ ]:
# STEP 27: TRANSACTION STATUS TEST

def test_transaction_status_is_valid():
    assert clean_df["transaction_status"].isin(
        ["Success", "Failed", "Pending"]
    ).all()

print("Transaction status test created successfully!")

Transaction status test created successfully!


In [10]:
# STEP 28: TRANSACTION DIRECTION TEST

def test_transaction_direction_is_valid():
    assert clean_df["transaction_direction"].isin(
        ["Debit", "Credit"]
    ).all()

print("Transaction direction test created successfully!")

Transaction direction test created successfully!


In [16]:
# IMPORT PYTEST AGAIN

import ipytest
ipytest.autoconfig()

print("Pytest is ready!")

Pytest is ready!


In [17]:
# STEP 29: RUN ALL TEST CASES

ipytest.run()

.                                                                                            [100%]
1 passed in 0.04s


<ExitCode.OK: 0>

In [19]:
# STEP 30: RESTORE ALL AUTOMATED TEST CASES

def test_positive_transaction_amounts():
    assert (clean_df["transaction_amount"] > 0).all()

def test_fraud_flags_are_valid():
    assert clean_df["is_fraud"].isin([0, 1]).all()

def test_transaction_dates_are_valid():
    assert clean_df["transaction_date"].notna().all()

def test_account_balance_is_valid():
    assert (clean_df["account_balance"] >= 0).all()

def test_credit_score_is_valid():
    assert clean_df["credit_score"].between(300, 900).all()

def test_transaction_ids_are_not_missing():
    assert clean_df["transaction_id"].notna().all()

def test_transaction_ids_are_unique():
    assert clean_df["transaction_id"].is_unique

def test_transaction_status_is_valid():
    assert clean_df["transaction_status"].isin(
        ["Success", "Failed", "Pending"]
    ).all()

def test_transaction_direction_is_valid():
    assert clean_df["transaction_direction"].isin(
        ["Debit", "Credit"]
    ).all()

print("All 9 test functions restored successfully!")

All 9 test functions restored successfully!


In [20]:
# STEP 31: RUN ALL AUTOMATED TESTS

ipytest.run()

........F                                                                                    [100%]
============================================ FAILURES =============================================
________________________________ test_transaction_status_is_valid _________________________________

    def test_transaction_status_is_valid():
>       assert clean_df["transaction_status"].isin(
            ["Success", "Failed", "Pending"]
        ).all()
E       AssertionError: assert np.False_
E        +  where np.False_ = all()
E        +    where all = 0         True\n1         True\n2         True\n3         True\n4         True\n          ... \n549995    True\n549996    True\n549997    True\n549998    True\n549999    True\nName: transaction_status, Length: 550000, dtype: bool.all
E        +      where 0         True\n1         True\n2         True\n3         True\n4         True\n          ... \n549995    True\n549996    True\n549997    True\n549998    True\n549999    True\nName: t

<ExitCode.TESTS_FAILED: 1>

In [21]:
# STEP 32: CHECK TRANSACTION STATUS VALUES

print(clean_df["transaction_status"].value_counts())

print("\nUnique Status Values:")
print(clean_df["transaction_status"].unique())

transaction_status
Success     506247
Failed       21905
Reversed     11019
Pending      10829
Name: count, dtype: int64

Unique Status Values:
['Success' 'Pending' 'Failed' 'Reversed']


In [22]:

# STEP 33: CORRECT TRANSACTION STATUS TEST

def test_transaction_status_is_valid():
    assert clean_df["transaction_status"].isin(
        ["Success", "Failed", "Pending", "Reversed"]
    ).all()

print("Transaction status test updated successfully!")

Transaction status test updated successfully!


In [23]:
# STEP 34: RUN ALL AUTOMATED TESTS

ipytest.run()

.........                                                                                    [100%]
9 passed in 0.18s


<ExitCode.OK: 0>

In [24]:
# STEP 35: TRANSACTION HOUR TEST

def test_transaction_hour_is_valid():
    assert clean_df["transaction_hour"].between(0, 23).all()

print("Transaction hour test created successfully!")

Transaction hour test created successfully!


In [25]:
# FINAL TEST EXECUTION

ipytest.run()

..........                                                                                   [100%]
10 passed in 0.15s


<ExitCode.OK: 0>